**Code-only release candidate.** Market inputs and executed outputs are deliberately absent pending redistribution clearance. See `DATA_SCHEMA.md` before running market-dependent cells. `python src/teaching_interfaces.py` runs deterministic synthetic checks without market data; it does not reproduce the empirical results.

# Portfolio Selection: Exact Search, Local Swap and QAOA

## Goal

Companion to Chapter 2, Section 2.2. Translate an equal-weight four-of-eight portfolio objective into QUBO coefficients and an Ising circuit. Separate exact simulator parameter training from finite-shot final decoding. Follow the equations before running the full case: means scale by 1/K, covariance by 1/K squared, and a penalty enforces K=4.

## Setup

Run this notebook from top to bottom. The experiment uses local simulation only; no cloud account, trading connection or API key is needed. Market observations stop on **31 August 2026**. The supplied data are quoted-price proxies, not verified total-return series.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

# Resolve only the package itself; no author-specific absolute path is needed.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data" / "daily_market.csv").exists(), "Authorised daily inputs are required; see DATA_SCHEMA.md. Synthetic checks run without market data via src/teaching_interfaces.py."
sys.path.insert(0, str(ROOT / "src"))
from common import DATA, RESULTS, market


## 1. Reproduce the monthly experiment

Each January--August 2026 decision uses 126 complete pre-decision return vectors. Risk aversion is 5. QAOA depths 1 and 2 each use three fixed seeds, at most 80 COBYLA evaluations and 2,048 final shots. This cell regenerates the results; it does not call a brokerage.

In [ ]:
from portfolio import run
summary = run()
display(pd.Series({k: summary[k] for k in ["dates", "runs", "optimal_runs", "failure_runs", "mean_feasibility", "mean_optimum_probability"]}, name="Results").to_frame())

## 2. Compare financial decisions

The objective gap is computed against exact feasible enumeration. It is not a return gap. The selected basket is the best observed feasible string, without substituting an exact optimum after a failed quantum run.

In [ ]:
runs = pd.read_csv(RESULTS / "portfolio_runs.csv")
display(runs.groupby(["method", "depth"]).agg(mean_gap=("gap", "mean"), mean_feasibility=("feasible_probability", "mean"), mean_seconds=("seconds", "mean")))
display(runs[(runs.method == "QAOA") & (runs.gap > 1e-10)][["month", "depth", "seed", "gap", "holdings"]])

## 3. Examine the sampling control

A tiny finite search space is easy to cover with many random samples. Compare against both unconstrained strings and uniform feasible baskets before interpreting a high best-of-many success rate.

In [ ]:
sampling = pd.read_csv(RESULTS / "portfolio_sampling.csv")
display(sampling.pivot(index="shots", columns="method", values="success").round(4))
display(Image(filename=str(RESULTS / "portfolio_comparison.png")))

## Interpretation and Next Steps

Use the displayed summary rather than a hard-coded success claim. Exact search evaluates only 70 feasible baskets, and uniform feasible sampling is a strong control. Ask which step changes if the penalty, mixer or depth changes. A high best-of-many success rate does not imply a large one-shot optimum probability. The return calculation is a stylised fixed-cohort price outcome with a 20-basis-point round-trip deduction, not evidence of persistent alpha.